# Putting It All Together

```{admonition} Overview
:class: tip

**Teaching:**

**Exercises:**

**Questions:**
  * 

**Objectives:**
  * 

```

## Create a VM

Since we only create resources as we need them in the cloud.  As an exercise you will now create a VM for our analysis.  In this case will give the VM **Full** access to **Storage**. 

Using the console navigate to the "Compute Engine" service and create a new VM with the following properties.
 * Call the VM "essentials"
 * Allow the VM "Full" access to "Storage".  This can be found under "Identity and API" and then selecting  "Set access for each API" and change "Storage" to "Full". **This will allow the VM to create, read, write, and delete all storage buckets in the project"**
 * Feel free to select a larger VM by changing the machine type to something larger, for example an "e2-standard-2".
 
When you are done connect to the machine as described below.

## Connect to the VM

First login to the instance from the Cloud Shell by running the following command:
```
gcloud compute ssh essentials
```
If prompted for a zone select `n` to find it automatically.  You can see an example session below.

```
learner@cloudshell:~ (just-armor-301114)$ gcloud compute ssh essentials
Did you mean zone [us-central1-b] for instance: [essentials] (Y/n)?  n

No zone specified. Using zone [us-west2-c] for instance: [essentials].
Linux essentials 4.19.0-18-cloud-amd64 #1 SMP Debian 4.19.208-1 (2021-09-29) x86_64

The programs included with the Debian GNU/Linux system are free software;
the exact distribution terms for each program are described in the
individual files in /usr/share/doc/*/copyright.

Debian GNU/Linux comes with ABSOLUTELY NO WARRANTY, to the extent
permitted by applicable law.
Last login: Tue Nov  9 20:12:49 2021 from 34.133.99.196
learner@essentials:~$
```


## Secure the VM

We first make sure that the VM is up to date with the latest security patches by running the following commands. Note: the `sudo unattended-upgrades` command only installs security patches.

In [6]:
sudo apt update
sudo unattended-upgrades

Hit:1 http://deb.debian.org/debian buster InRelease
Hit:2 http://deb.debian.org/debian buster-updates InRelease                    
Hit:3 http://deb.debian.org/debian buster-backports InRelease                  
Hit:4 http://security.debian.org/debian-security buster/updates InRelease      
Hit:5 http://packages.cloud.google.com/apt cloud-sdk-buster InRelease
Hit:6 http://packages.cloud.google.com/apt google-cloud-packages-archive-keyring-buster InRelease
Hit:7 http://packages.cloud.google.com/apt google-compute-engine-buster-stable InRelease
Reading package lists... Done
Building dependency tree       
Reading state information... Done
5 packages can be upgraded. Run 'apt list --upgradable' to see them.


## Get Example Code

We will now install `git` and use it to download the example code into your home directory.  For those of you who are unfamiliar with git, it is a way to collaboratively manage files and we will only be using it to download the example that we will be using. 

In [1]:
sudo apt-get install --yes git

Reading package lists... Done
Building dependency tree       
Reading state information... Done
git is already the newest version (1:2.20.1-2+deb10u3).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [2]:
cd ~

In [3]:
git clone https://github.internet2.edu/CLASS/CLASS-Examples.git

Cloning into 'CLASS-Examples'...
remote: Enumerating objects: 66, done.
remote: Total 66 (delta 0), reused 0 (delta 0), pack-reused 66
Unpacking objects: 100% (66/66), done.


We now change the current directory to the `landsat` directory in the `CLASS-Examples` directory that was just created by the previous git command.

In [4]:
cd ~/CLASS-Examples/landsat/

Your prompt should now change showing the current directory as follows.
```
learner@essentials:~/CLASS-Examples/landsat$
```

In [39]:
ls -l

total 36
-rw-r--r-- 1 learner learner  960 Dec 16 16:27 ReadMe.md
-rw-r--r-- 1 learner learner   72 Dec 16 16:27 clean.sh
-rw-r--r-- 1 learner learner  613 Dec 16 16:27 combine.py
drwxr-xr-x 4 learner learner 4096 Dec 16 16:37 data
-rw-r--r-- 1 learner learner  280 Dec 16 16:27 download.sh
-rw-r--r-- 1 learner learner  314 Dec 16 16:27 get-index.sh
drwxr-xr-x 2 learner learner 4096 Dec 16 16:43 output
-rw-r--r-- 1 learner learner   76 Dec 16 16:27 search.json
-rw-r--r-- 1 learner learner  783 Dec 16 16:27 search.py


## Access the bucket

First test that our tools are working and that we can access the public bucket that we will be using.

In [6]:
gsutil ls gs://gcp-public-data-landsat/

gs://gcp-public-data-landsat/index.csv.gz
gs://gcp-public-data-landsat/LC08/
gs://gcp-public-data-landsat/LE07/
gs://gcp-public-data-landsat/LM01/
gs://gcp-public-data-landsat/LM02/
gs://gcp-public-data-landsat/LM03/
gs://gcp-public-data-landsat/LM04/
gs://gcp-public-data-landsat/LM05/
gs://gcp-public-data-landsat/LO08/
gs://gcp-public-data-landsat/LT04/
gs://gcp-public-data-landsat/LT05/
gs://gcp-public-data-landsat/LT08/


The index file is a list of all the files in the bucket and we can use it to search and filter files.

We will get the index and uncompress the file placing it in the `data/` directory (this is ignored by git). This should take around 2 min with a `e2-medium` instance in the `us-west2` region.

In [7]:
mkdir -v data

mkdir: created directory 'data'


In [8]:
gsutil cp gs://gcp-public-data-landsat/index.csv.gz data/

Copying gs://gcp-public-data-landsat/index.csv.gz...
==> NOTE: You are downloading one or more large file(s), which would            
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

\ [1 files][760.9 MiB/760.9 MiB]   50.8 MiB/s                                   
Operation completed over 1 objects/760.9 MiB.                                    


We will now uncompress the index file to make it easier to use.  This may take some time depending on the machine type you are using.

In [9]:
gzip -df data/index.csv.gz

Again, check what happened.

In [10]:
ls -lh data

total 2.6G
-rw-r--r-- 1 learner learner 2.6G Dec 16 16:32 index.csv


We will now explore the data.  The `head` command simply displays the first few lines of the file.

In [11]:
head --lines=4 data/index.csv

SCENE_ID,PRODUCT_ID,SPACECRAFT_ID,SENSOR_ID,DATE_ACQUIRED,COLLECTION_NUMBER,COLLECTION_CATEGORY,SENSING_TIME,DATA_TYPE,WRS_PATH,WRS_ROW,CLOUD_COVER,NORTH_LAT,SOUTH_LAT,WEST_LON,EAST_LON,TOTAL_SIZE,BASE_URL
LC81030672020252LGN00,LC08_L1GT_103067_20200908_20200918_01_T2,LANDSAT_8,OLI_TIRS,2020-09-08,01,T2,2020-09-08T01:04:03.7134200Z,L1GT,103,67,44.99,-9.07156,-11.17811,134.61622,136.70979,1009724166,gs://gcp-public-data-landsat/LC08/01/103/067/LC08_L1GT_103067_20200908_20200918_01_T2
LC81590702021230LGN00,LC08_L1TP_159070_20210818_20210826_01_T1,LANDSAT_8,OLI_TIRS,2021-08-18,01,T1,2021-08-18T06:51:18.8060269Z,L1TP,159,70,15.72,-13.40348,-15.51165,47.1186,49.24972,989147313,gs://gcp-public-data-landsat/LC08/01/159/070/LC08_L1TP_159070_20210818_20210826_01_T1
LE70620122002162PAC00,LE07_L1TP_062012_20020611_20170129_01_T1,LANDSAT_7,ETM,2002-06-11,01,T1,2002-06-11T20:17:05.6581076Z,L1TP,62,12,41.0,69.38475,67.14636,-136.16562,-129.66669,224906243,gs://gcp-public-data-landsat/LE07/01/062/012

## Search for Data

We can see the data is well formed and what we expect.  We will now use this data to download data related to a specific point and for the Landsat 8.  The following script does a simple filter.

In [12]:
cat search.py

#!/usr/bin/python3
import json
import csv
import sys

# Example: Burr Oak Tree
# 38.899313,-92.464562 (Lat north+, Long west-) ; Landsat Path 025, Row 033
config=json.load(open("search.json"))
lat,lon=config['lat'],config['lon']
landsat=config['landsat']

reader=csv.reader(sys.stdin)
header=next(reader) # skip header
for l in reader:
    SCENE_ID,PRODUCT_ID,SPACECRAFT_ID,SENSOR_ID,DATE_ACQUIRED,COLLECTION_NUMBER,COLLECTION_CATEGORY,SENSING_TIME,DATA_TYPE,WRS_PATH,WRS_ROW,CLOUD_COVER,NORTH_LAT,SOUTH_LAT,WEST_LON,EAST_LON,TOTAL_SIZE,BASE_URL=l
    west,east=float(WEST_LON),float(EAST_LON)
    north,south=float(NORTH_LAT),float(SOUTH_LAT)
    if SPACECRAFT_ID==landsat and north >= lat and south <= lat and west <= lon and east >= lon:
        print(BASE_URL) # output BASE_URL


We can see that the actual search data comes from the file `search.json`.  The program reads the data from the standard input and iterates over all rows in the CSV file.  It filters the results for which the image contains the pint and prints out the bucket URL for them. We are interested in all products that contain the Burr Oak Tree.

In [13]:
cat search.json

{
    "lat": 38.899313,
    "lon": -92.464562,
    "landsat": "LANDSAT_8"
}


Now lets test this on a subset of the data.

In [14]:
head --lines=200000 data/index.csv | python3 search.py

gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1
gs://gcp-public-data-landsat/LO08/01/025/033/LO08_L1TP_025033_20201108_20201120_01_T1
gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1GT_025033_20200109_20200114_01_T2
gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20150706_20180131_01_T1
gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20170711_20170726_01_T1


## Download the Data

Now that we have a list of folders we are interested, we will now download them with a simple script that takes bucket addresses (URL's) and downloads them with the `gsutil` program.

In [15]:
cat download.sh

#!/bin/bash

# Read space separated URL from STDIN and download 
while read -r URL ; do
    echo "+++ $URL"
    # -m parallel
    # -n no-clobber (do not re-download data)
    # -r recursive (download all the data in the specified URL)
    gsutil -m cp -n -r "${URL}/" data/
done


Get the first 2 datasets

In [16]:
head --lines=200000 data/index.csv | python3 search.py | head --lines=2 | bash download.sh

+++ gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1
Copying gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1/LC08_L1TP_025033_20140124_20170306_01_T1_ANG.txt...
Copying gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1/LC08_L1TP_025033_20140124_20170306_01_T1_B11.TIF...
Copying gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1/LC08_L1TP_025033_20140124_20170306_01_T1_B3.TIF...
Copying gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1/LC08_L1TP_025033_20140124_20170306_01_T1_B1.TIF...
Copying gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1/LC08_L1TP_025033_20140124_20170306_01_T1_B10.TIF...
Copying gs://gcp-public-data-landsat/LC08/01/025/033/LC08_L1TP_025033_20140124_20170306_01_T1/LC08_L1TP_025033_20140124_20170306_01_T1_B4.TIF...
Copying gs://gcp-public-data-landsat/

Check that the data was downloaded

In [17]:
ls -l data

total 2665520
drwxr-xr-x 2 learner learner       4096 Dec 16 16:37 LC08_L1TP_025033_20140124_20170306_01_T1
drwxr-xr-x 2 learner learner       4096 Dec 16 16:37 LO08_L1TP_025033_20201108_20201120_01_T1
-rw-r--r-- 1 learner learner 2729479404 Dec 16 16:32 index.csv


## Processing the Data

We will now use simple script to combine the color bands and export it as a PNG

In [18]:
cat combine.py

#!/usr/bin/python3
import os
import rasterio

# Open the first directory in data, could walk entire tree
for dirname, dirs, files in os.walk('data'):
    source = dirs[0]
    break

# Open band (B2/Blue) and copy metadata for result.
with rasterio.open("data/%s/%s_B2.TIF" % (source, source)) as band2:
    meta = band2.meta

# Combine bands into PNG
meta.update(count = 3, driver='PNG')
with rasterio.open("output/result-%s.png" % source, 'w+', **meta) as output:
    for i in range(1, 4):
        print(source, i)
        output.write_band(i,rasterio.open("data/%s/%s_B%d.TIF" % (source, source, i+3)).read(1))


In [27]:
python3 combine.py
/bin/true # ignore this line used for jupyter

Traceback (most recent call last):
  File "combine.py", line 3, in <module>
    import rasterio
ModuleNotFoundError: No module named 'rasterio'


Oops, let's install the library (note: the output will be slightly different due to how this Lesson is built)
and while we are at it we also need to create the output directory.

In [28]:
mkdir -v output/
sudo apt-get install python3-rasterio --yes

install: creating directory 'output/'
Reading package lists... Done
Building dependency tree       
Reading state information... Done
python3-rasterio is already the newest version (1.0.21-1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [29]:
/usr/bin/python3 combine.py

LO08_L1TP_025033_20201108_20201120_01_T1 1
LO08_L1TP_025033_20201108_20201120_01_T1 2
LO08_L1TP_025033_20201108_20201120_01_T1 3
ERROR 4: output/result-LO08_L1TP_025033_20201108_20201120_01_T1.png: No such file or directory


Note, the "ERROR 4" is expected.

Verify the results were created.

In [30]:
ls -lh output

total 196M
-rw-r--r-- 1 learner learner 196M Dec 16 16:43 result-LO08_L1TP_025033_20201108_20201120_01_T1.png
-rw-r--r-- 1 learner learner  910 Dec 16 16:43 result-LO08_L1TP_025033_20201108_20201120_01_T1.png.aux.xml


## Exporting the Results

Now that we have the output data we will create a bucket to place the results. We will first create a bucket with a reasonable set of options.

We fisrt store the bucket name in the `BUCKET` environment variable for future use.  This time we will specify a realistic set of options for a private bucket used for computation.

Options (run `gsutil mb --help` for more information):
 * `-b on` specifies uniform bucket-level access.
 * `-l $REGION` puts the data in a specific region for lower cost and lower latency.
 * `--pap enforced` turns on public access prevention to help keep data private.  
 
The uniform bucket level access (Bucket Policy Only enabled: true) puts the data access permissions (ACL) on the entire bucket, not on each object in the bucket.  This makes the permissions obvious and makes security much more predictable.
 
As usual, we must set our environment.  In this case we also set a `REGION` environment variable to indicate where in the world we want the data to be stored.


In [31]:
BUCKET="essentials-${USER}-$(date +%F)"
REGION="us-west2"
echo "bucket: $BUCKET region: $REGION"

bucket: essentials-learner-2021-12-16 region: us-west2


In [32]:
gsutil mb -b on -l $REGION --pap enforced "gs://$BUCKET"

Creating gs://essentials-learner-2021-12-16/...


Now copy the output data to the bucket. The `-r` flag recursively copies the output directory and `-m` copies the files in parallel.  Note the locations of the `-m` and `-r` switches as they apply globally and to the `cp` command respectively.

In [35]:
gsutil -m cp -r "output" "gs://$BUCKET"

Copying file://output/result-LO08_L1TP_025033_20201108_20201120_01_T1.png [Content-Type=image/png]...
Copying file://output/result-LO08_L1TP_025033_20201108_20201120_01_T1.png.aux.xml [Content-Type=application/xml]...
==> NOTE: You are uploading one or more large file(s), which would run          
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

| [2/2 files][195.7 MiB/195.7 MiB] 100% Done                                    
Operation compl

Verify that the output was uploaded.

In [37]:
gsutil ls "gs://$BUCKET"

gs://essentials-learner-2021-12-16/output/


In [38]:
gsutil ls -lh "gs://$BUCKET/output"

195.71 MiB  2021-12-16T16:45:52Z  gs://essentials-learner-2021-12-16/output/result-LO08_L1TP_025033_20201108_20201120_01_T1.png
     910 B  2021-12-16T16:45:50Z  gs://essentials-learner-2021-12-16/output/result-LO08_L1TP_025033_20201108_20201120_01_T1.png.aux.xml
TOTAL: 2 objects, 205222402 bytes (195.72 MiB)


## Viewing the Results

You now can view the results by using the Google Cloud Platform Web Console and navigating to "Cloud Storage", selecting the bucket, and then the result object you wish to view (select the `.png` file).  You will need to click the "Preview" button given the large size of the image.

## Cleanup

We will now leave the resources running in order to learn more about monitoring costs and will clean up all the resources as the end of Lesson.  **Don't forget to do this!**